<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/alternative_turn_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
!pip install -U dspy

In [ ]:
import pandas as pd
import dspy

In [ ]:
lm = dspy.LM('ollama_chat/llama3.2:1b', api_base = 'http://localhost:11434', api_key='3cfbbf94908c439cb71556eaa98cd4b6.MDioHrLPBO8VqIhNLP4tfAsP')
dspy.configure(lm=lm)

In [ ]:
from typing import Literal


class TurnClassifier(dspy.Signature):
  "Classify Dungeons & Dragons gameplay turns."
  context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
  turn: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
  category: Literal['Purposeful Roleplay',
                      'Cooperative Roleplay',
                      'Open-ended Roleplay',
                      'Disruptive Roleplay',
                      'Rules Query',
                      'Dice Rolls',
                      'Simple Response',
                      'Game State Information',
                      'Prompting Game Master',
                      'Humour',
                      'Chit-Chat',
                      'Simple Response',
                      'Game State Update',
                      'Rules Clarification',
                      'NPC Action',
                      'Out-of-character Game Master'] = dspy.OutputField()

classify = dspy.Predict(TurnClassifier)

In [ ]:
system_message = dspy.ChatAdapter().format_system_message(TurnClassifier)
print(system_message)

In [ ]:
def validate_category(example, prediction, trace=None):
    return prediction.category == example.category

In [ ]:
training_df = pd.read_excel('Dungeons-and-Dragons-Turn-Classification/trainset_annotator_1.xlsx')

In [ ]:
training_df.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
trainset = []

for context, current_turn, category in training_df.values:
    trainset.append(dspy.Example(context=context, turn=current_turn, category=category).with_inputs("context", "turn", "category"))

In [ ]:
def classify_turn(context, turn):
    try:
        prediction = classify(context=context, turn=turn)
        return prediction.category
    except Exception as e:
        return 0

In [ ]:
predicted_categories = []

predicted_df = pd.read_excel('predicted_categories.xlsx')
predicted_df.drop(columns=['Unnamed: 0'], inplace=True)

for category in predicted_df.values:
  predicted_categories.append(dspy.Example(category=category))

# for i in range(len(
# for i in range(len(trainset)):
#   predicted_categories.append(classify_turn(trainset[i]['context'], trainset[i]['turn']))


In [ ]:
predicted_categories

In [ ]:
# df = pd.DataFrame(predicted_categories, columns=['category'])

In [ ]:
# df.to_excel('predicted_categories.xlsx')

In [ ]:
from dspy.evaluate import Evaluate

evaluator = Evaluate(devset=trainset, num_threads=1, display_progress=True, display_table=5)
# evaluator(classify, metric=validate_category)

In [ ]:
trainset

In [ ]:
import dspy.teleprompt as tp

tp = dspy.MIPROv2(metric=validate_category, auto="light")
optimized_classify = tp.compile(classify, trainset=trainset, max_labeled_demos=0, max_bootstrapped_demos=0)